In [1]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
import time
from datetime import timedelta
#倒入mnist数据集，注意mnist是一个大杂烩，需要导入数据特征和标签
X_mnist, y_mnist = fetch_openml('mnist_784', return_X_y=True, as_frame=False,
                                parser='auto')

X_train,X_valid,X_test = X_mnist[:50000],X_mnist[50000:60000],X_mnist[60000:70000]
y_train,y_valid,y_test = y_mnist[:50000],y_mnist[50000:60000],y_mnist[60000:70000]
#我这里数据进行了一定的处理，加快计算速度
X_train_scaled = X_train / 255.0
X_valid_scaled = X_valid / 255.0
X_test_scaled = X_test / 255.0


In [2]:
rfc = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
etc = ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1)
svc = SVC(probability=True,random_state=42)
knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
models = [
    ('RFM',rfc),
    ('EFC',etc),
    ('SVM',svc),
    ('KNN',knn)
]

In [3]:
#这一个code是为了最后计算准确性来转化的，将其提到第一个code也可以
y_train = y_train.astype(np.int64)
y_valid = y_valid.astype(np.int64)
y_test = y_test.astype(np.int64)


In [4]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

In [5]:
final_learner = LogisticRegression(random_state=42)

In [6]:
stacking_clf = StackingClassifier(
    estimators=models,
    final_estimator=final_learner,
    cv=3,  # number of cross-validation folds
    n_jobs=-1,
    stack_method='predict_proba'
)
stacking_clf.fit(X_train_scaled,y_train)
y_pre = stacking_clf.predict(X_valid_scaled)
score = accuracy_score(y_valid,y_pre)
print(score)

0.9833


In [7]:
# 最终测试集评估
y_test_pred2 = stacking_clf.predict(X_test_scaled)
final_score = accuracy_score(y_test.astype(int), y_test_pred2.astype(int))

print(f"🎉 stacking模型最终测试集准确率: {final_score:.4f}")

🎉 stacking模型最终测试集准确率: 0.9805


以下为额外的内容，不是必须

In [8]:
#以下是保存为pkl的代码，可以不保存
import joblib
import os

# 1. 定义文件名 (保存到当前目录)
model_filename = "mnist_stacking_9833_final.pkl"

# 2. 执行保存
print(f"正在保存模型到当前目录: {os.getcwd()} ...")
joblib.dump(stacking_clf, model_filename)

# 3. 验证保存结果
if os.path.exists(model_filename):
    file_size = os.path.getsize(model_filename) / (1024 * 1024)
    print("="*40)
    print(f"✅ 保存成功！")
    print(f"文件位置: {os.path.join(os.getcwd(), model_filename)}")
    print(f"文件大小: {file_size:.2f} MB")
    print("="*40)
else:
    print("❌ 保存失败，请检查权限或路径。")

正在保存模型到当前目录: /Users/sususu/Documents/my_ai_project/Machine Learning /07_ensemble_learning_and_random_forests.ipynb   ...
✅ 保存成功！
文件位置: /Users/sususu/Documents/my_ai_project/Machine Learning /07_ensemble_learning_and_random_forests.ipynb  /mnist_stacking_9833_final.pkl
文件大小: 734.79 MB
